In [0]:
%sql
select * from zara_lakehouse.retail_silver.fact_sales_silver

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
gold_fact_sales_target = "zara_lakehouse.retail_gold.gold_fact_sales"

In [0]:
%sql
SELECT
    fs.product_id,
    fs.city,
    
    fs.date,

    -- aggregated measures
    SUM(fs.pieces_sold) AS total_pieces_sold,
    ROUND(SUM(fs.revenue),2) AS total_revenue,
    ROUND(SUM(fs.discount_amount),2) AS total_discount,
    ROUND(SUM(fs.net_revenue),2) AS total_net_revenue,

    -- useful metrics
    AVG(fs.unit_price) AS avg_price,
    AVG(fs.discount) AS avg_discount

FROM zara_lakehouse.retail_silver.fact_sales_silver fs
GROUP BY fs.product_id, fs.city, fs.date;

In [0]:
df = spark.sql("""
SELECT
    fs.product_id,
    fs.city,
    fs.date,

    -- aggregated measures
    SUM(fs.pieces_sold) AS total_pieces_sold,
    ROUND(SUM(fs.revenue),2) AS total_revenue,
    ROUND(SUM(fs.discount_amount),2) AS total_discount,
    ROUND(SUM(fs.net_revenue),2) AS total_net_revenue,

    -- useful metrics
    AVG(fs.unit_price) AS avg_price,
    AVG(fs.discount) AS avg_discount

FROM zara_lakehouse.retail_silver.fact_sales_silver fs
GROUP BY fs.product_id, fs.city, fs.date;     
               """)

In [0]:
df = df.write \
        .mode("overwrite") \
        .option ("mergeSchema", "true") \
        .saveAsTable(gold_fact_sales_target)

In [0]:
%sql
select * from zara_lakehouse.retail_gold.gold_fact_sales